# Question 1: SQL
Imagine the dataset you received is a table in a database with the name “cohort_data”.Write a SQL query to return a table that contains the Retention Rate for Day 1, Day 2, Day 3 and so on, up to Day 7.

In [10]:
import pandas as pd
data = pd.read_csv("(local path)")
data

,user_id,acquisition_date,date,is_acquired,ab_group,ad_revenue,ad_revenue_interstitial,ad_revenue_rewarded,ad_impressions_interstitial,ad_impressions_rewarded
0,2788165294615712,2022-10-09,2022-10-13,False,test_group,0.000000,0.000000,0.0,0,0
1,5793823020638738,2022-10-17,2022-10-17,False,control_group,0.000000,0.000000,0.0,0,0
2,5140934416008790,2022-10-16,2022-10-16,False,control_group,0.000000,0.000000,0.0,0,0
3,6118659741496606,2022-10-11,2022-10-14,False,control_group,0.000000,0.000000,0.0,0,0
4,5444473795665560,2022-10-08,2022-10-13,True,control_group,0.000000,0.000000,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...
189321,5493008420778614,2022-10-07,2022-10-07,True,test_group,0.079007,0.079007,0.0,1,0
189322,6193547923992724,2022-10-20,2022-10-20,False,test_group,0.073664,0.073664,0.0,1,0
189323,3086649111378801,2022-10-16,2022-10-16,False,test_group,0.000000,0.000000,0.0,0,0
189324,1375537162571306,2022-10-09,2022-10-10,False,control_group,0.344881,0.344881,0.0,5,0


In [11]:
from sqlalchemy import create_engine

def generate_table(df, db_path, table_name):
    
    # Create a SQLAlchemy engine
    engine = create_engine(db_path)
    
    # Write the DataFrame to a SQL table
    df.to_sql(table_name, engine, if_exists='replace', index=False)

db_path = 'sqlite:///cohort_data.db'  # database URL
table_name = 'cohort_data'  # table name

table = generate_table(data, db_path, table_name)


In [12]:

def query_data(db_path, query):
    # Connect to the SQLite database
    engine = create_engine(db_path)
    
    # Execute the query and fetch the results
    df = pd.read_sql_query(query, con=engine)
    
    
    return df

query = \
""" 

WITH base as (SELECT 
acquisition_date as install_date,
date as event_date,
CAST(julianday(date) - julianday(acquisition_date) AS INT) as cohort_day,
COUNT(DISTINCT user_id) as user_count
FROM cohort_data 
GROUP BY install_date, event_date, cohort_day),

final as (SELECT
    install_date,
     sum(user_count) filter (where cohort_day = 0) as day_0,
     sum(user_count) filter (where cohort_day = 1) as day_1,
     sum(user_count) filter (where cohort_day = 1) as day_2,
     sum(user_count) filter (where cohort_day = 3) as day_3,
     sum(user_count) filter (where cohort_day = 4) as day_4,
     sum(user_count) filter (where cohort_day = 5) as day_5,
     sum(user_count) filter (where cohort_day = 6) as day_6,
     sum(user_count) filter (where cohort_day = 7) as day_7
FROM 
    base
GROUP BY 
    install_date
ORDER BY
    install_date)

SELECT
    install_date,
    ROUND((CAST(day_0 as FLOAT) / CAST(day_0 as FLOAT)) * 100,2) as day_0,
    ROUND((CAST(day_1 as FLOAT) / CAST(day_0 as FLOAT)) * 100,2) as day_1,
    ROUND((CAST(day_2 as FLOAT) / CAST(day_0 as FLOAT)) * 100,2) as day_2,
    ROUND((CAST(day_3 as FLOAT) / CAST(day_0 as FLOAT)) * 100,2) as day_3,
    ROUND((CAST(day_4 as FLOAT) / CAST(day_0 as FLOAT)) * 100,2) as day_4,
    ROUND((CAST(day_5 as FLOAT) / CAST(day_0 as FLOAT)) * 100,2) as day_5,
    ROUND((CAST(day_6 as FLOAT) / CAST(day_0 as FLOAT)) * 100,2) as day_6,
    ROUND((CAST(day_7 as FLOAT) / CAST(day_0 as FLOAT)) * 100,2) as day_7
FROM
    final
GROUP BY 
    install_date


"""
query_result = query_data(db_path, query)
query_result

,install_date,day_0,day_1,day_2,day_3,day_4,day_5,day_6,day_7
0,2022-10-07,100.0,14.85,14.85,9.09,7.44,6.82,5.75,5.41
1,2022-10-08,100.0,16.14,16.14,8.41,7.12,6.50,6.39,5.48
2,2022-10-09,100.0,15.09,15.09,8.56,6.77,6.46,5.33,5.78
3,2022-10-10,100.0,15.11,15.11,9.85,7.26,6.16,6.08,5.46
4,2022-10-11,100.0,15.35,15.35,8.31,6.44,6.28,5.30,3.82
5,2022-10-12,100.0,15.23,15.23,7.71,6.77,5.85,3.99,3.99
6,2022-10-13,100.0,14.63,14.63,7.15,5.83,4.21,3.61,2.90
7,2022-10-14,100.0,13.23,13.23,6.23,4.45,3.82,3.06,3.14
8,2022-10-15,100.0,12.83,12.83,5.06,4.11,3.56,3.16,NaN
9,2022-10-16,100.0,12.61,12.61,5.81,4.27,3.89,NaN,NaN
